In [ ]:
import sys
sys.path.insert(0, "./local_libs_clean")

In [ ]:
import os

# Change to your token
os.environ["HUGGINGFACE_HUB_TOKEN"] = ""

In [ ]:
from huggingface_hub import login

# Change to your token
login(token="")

In [ ]:
%pip install torch intel_extension_for_pytorch
%pip install transformers accelerate torch PyYAML
%pip install -U bitsandbytes

여기서 커널 restart하는거 추천 (kernel restart recommended)

In [ ]:
import transformers

#check transformers's version
print(transformers.__version__)

In [ ]:
import json
import torch
import intel_extension_for_pytorch as ipex
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig

In [ ]:
#  PyTorch에서 XPU가 감지되는지 확인 True가 나와야함
# Check if pytorch can sense XPU (intel GPU)
print(torch.xpu.is_available()) 

In [ ]:
# Choose your model:
# model_name = "mistralai/Mistral-7B-Instruct-v0.1" #TOO BIG
# model_name = "stabilityai/stablelm-base-alpha-3b" # STILL TOO BIG
model_name = "EleutherAI/gpt-neo-1.3B"
# model name = "microsoft/Phi-3-medium-4k-instruct"
# model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="xpu" # if you are going to use GPU
    #device map="auto" 
)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
class InstructDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=512):  # lowered max_length
        self.data = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        record = self.data[idx]
        instruction = record.get("instruction", "").strip()
        input_text = record.get("input", "").strip()
        output_text = record.get("output", "").strip()
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nResponse: "
        full_text = prompt + output_text

        # Tokenize the full text with the new max_length.
        tokenized_full = self.tokenizer(full_text, truncation=True, max_length=self.max_length, return_tensors="pt")
        input_ids = tokenized_full["input_ids"].squeeze(0)
        attention_mask = tokenized_full["attention_mask"].squeeze(0)

        # Tokenize only the prompt to calculate the length.
        tokenized_prompt = self.tokenizer(prompt, truncation=True, max_length=self.max_length, return_tensors="pt")
        prompt_length = tokenized_prompt["input_ids"].squeeze(0).shape[0]

        labels = input_ids.clone()
        labels[:prompt_length] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
def custom_collate_fn(batch):
    # Pad sequences to the same length.
    input_ids = [x["input_ids"] for x in batch]
    attention_masks = [x["attention_mask"] for x in batch]
    labels = [x["labels"] for x in batch]

    input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks_padded = pad_sequence(attention_masks, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids_padded,
        "attention_mask": attention_masks_padded,
        "labels": labels_padded
    }

In [ ]:
%pip install peft
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
# Check the target modules.
for name, module in model.named_modules():
    print(name)

In [ ]:
lora_config = LoraConfig(
    r=8,                # 업데이트할 low-rank 차원 (예: 8)
    lora_alpha=16,      # 스케일링 팩터
    lora_dropout=0.05,  # 드롭아웃 확률
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],  # 위에 코드 output을 gpt에 붙여넣기한 후 어떻게 수정해야하는지 질문 후 모델 아키텍처에 맞게 수정 
    task_type=TaskType.CAUSAL_LM,
)

# 4. LoRA 어댑터 주입 (이 시점에서 모델은 full precision이므로 LoRA가 정상 작동함)
model = get_peft_model(model, lora_config)

# 5. (선택 사항) 학습 가능한 파라미터 확인
model.print_trainable_parameters()

In [ ]:
import torch.optim as optim

# IPEX - INFO - Currently split master weight for xpu only support sgd
optimizer = optim.SGD(
    model.parameters(), 
    lr=3e-3,            # Higher LR since LoRA updates a small fraction of params
    momentum=0.9,       # Stabilizes training
    weight_decay=1e-3   # Helps regularize the model
)

# IPEX 최적화 시 optimizer 인자 추가
# For Adam Optimizer:
# model, optimizer = ipex.optimize(model, optimizer=optimizer, dtype=torch.bfloat16)

# For SGD optimizer:
# Ensure FP32 precision for IPEX optimization
model, optimizer = ipex.optimize(model, optimizer=optimizer, dtype=torch.float32)

In [ ]:
# Create the dataset from your preprocessed file.
dataset_file = "validation_data_1.jsonl"
train_dataset = InstructDataset(dataset_file, tokenizer, max_length=512)

In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_gpt-neo-1.3B",  # decide output directory name
    num_train_epochs=3,                      # You can increase this if you wish
    per_device_train_batch_size=4,           # Increased from 1 to 4
    gradient_accumulation_steps=1,           # Reduced accumulation due to larger batch size
    learning_rate=5e-5,                      # You may experiment with a slightly higher rate
    fp16=True,                               # Enable fp16 if supported by your GPU
    bf16=False,                              # or bf16 if that's better for your hardware
    logging_steps=10,
    save_steps=100,                          # Adjust saving frequency based on training speed
    gradient_checkpointing=False,            # Disable if memory is no longer a bottleneck
    # optim = "adamw_torch"
    # optim="sgd",
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_steps=10000,                          # Increase for a full training run
    save_total_limit=2,
)

In [ ]:
# Instantiate the Trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=custom_collate_fn,
    tokenizer=tokenizer,
)

여기는 우리 데이터가 fine tuning에 적절한 구조로 되어있나 한번 더 확인하는 과정

In [ ]:
# Print a few samples to inspect the prompt and labels.
for i in range(3):
    sample = train_dataset[i]
    prompt_text = tokenizer.decode(sample["input_ids"])
    # For labels, tokens with -100 are masked out.
    label_tokens = [tok for tok in sample["labels"].tolist() if tok != -100]
    response_text = tokenizer.decode(label_tokens)
    
    print(f"Sample {i}:")
    print("Prompt + Response:")
    print(prompt_text)
    print("Extracted Response (for loss):")
    print(response_text)
    print("="*50)

In [ ]:
batch = [train_dataset[i] for i in range(3)]
collated = custom_collate_fn(batch)
print("Batch input_ids shape:", collated["input_ids"].shape)
print("Batch attention_mask shape:", collated["attention_mask"].shape)
print("Batch labels shape:", collated["labels"].shape)

In [ ]:
with open("counsel_chats_instruct_finetune_data.jsonl", "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        record = json.loads(line)
        if not all(k in record and record[k].strip() for k in ["instruction", "input", "output"]):
            print(f"Record {idx} is missing required fields or is empty")


In [ ]:
if tokenizer.pad_token is None:
    tokenizerQpad_token = tokenizer.eos_token
print("Pad token id:", tokenizer.pad_token_id)

In [ ]:
trainer.train()

Training loss values over 1000 steps fluctuate between 2.5 and 2.8, but they do not show a clear downward trend. Needs to be improved

# Now Let's see how fine tuned model differ from original model
## Test fine-tuned model: Q."what should I do when I feel depressed?"

In [ ]:
# Load the fine-tuned model
model_path = "./fine_tuned_gpt-neo-1.3B/checkpoint-10000"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Test inference
#input_text = "I feel overwhelmed and anxious. I don’t know if therapy will actually help me. What do you think?"
input_text = "i think i should go grab a bottle now and get some dancing music on"
inputs = tokenizer(input_text, return_tensors="pt")

# Generate response
outputs = model.generate(
    **inputs,
    max_length=100,
    do_sample=True,      # Enables random sampling
    temperature=0.7,     # Controls randomness (lower = more deterministic)
    top_k=50,            # Limits choices to top-k likely words
    top_p=0.9,           # Nucleus sampling (filters low-probability words)
    repetition_penalty=1.2  # Reduces repetitive text
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Fine-Tuned AI Response:", response)

## Test original model: Q."what should I do when I feel depressed?"

In [ ]:
# Load the original GPT-Neo 1.3B model
model_name = "EleutherAI/gpt-neo-1.3B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Example input prompt
input_text = "i think i should go grab a bottle now and get some dancing music on"
# Tokenize input
inputs = tokenizer(input_text, return_tensors="pt")
# Generate response
outputs = model.generate(
    **inputs,
    max_length=100,
    do_sample=True,      # Enables random sampling
    temperature=0.7,     # Controls randomness (lower = more deterministic)
    top_k=50,            # Limits choices to top-k likely words
    top_p=0.9,           # Nucleus sampling (filters low-probability words)
    repetition_penalty=1.2  # Reduces repetitive text
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print the generated response
print("GPT-Neo Response:", response)
